# B300 한 장에서 추론 소프트웨어와 성능 측정

**질문 하나에 답하기**와 **여럿이 동시에 질문했을 때 답하기**는 무엇이 다를까? 먼저 같은 Qwen 모델을 Transformers와 vLLM으로 직접 실행한다. 이어서 vLLM 서버를 켜고, 질문 수와 질문·답변 길이를 바꿔 얼마나 기다리는지 잰다.

| 발표에서 배운 내용 | 노트북에서 확인할 곳 |
| --- | --- |
| CUDA 커널은 GPU에서 실행하는 함수다. PyTorch는 이런 함수를 호출해 모델을 계산한다. | 2절에서 같은 입력을 처리하는 두 프로그램의 코드를 읽는다. [발표의 커널 예제](chapter-04.html#cuda) |
| 추론 엔진은 어떤 질문들을 함께 계산할지 정한다. 함께 계산하는 묶음을 배치라고 한다. | 3절에서 vLLM 서버를 켜고 질문 하나에 답을 받아 본다. [발표의 배치 예제](chapter-04.html#batch) |
| 부하 테스트는 여러 요청을 보내 서버 전체의 처리량과 각 사용자의 대기 시간을 재는 실험이다. | 5~7절에서 한 번에 하나의 조건을 바꿔 비교한다. [발표의 측정 구간](chapter-04.html#metrics) |

**실행 위치**는 Linux 서버 또는 컨테이너에서 B300 한 장이 보이는 환경이다. 직접 실행과 서버 실험 모두 같은 GPU 한 장을 쓴다. B300을 지원하는 PyTorch·CUDA·vLLM이 필요하다. 이 폴더를 작업 디렉터리로 열고 vLLM 환경의 Python 커널을 선택한다. 두 Python 파일은 노트북과 같은 폴더에 둔다.

**실행 순서**는 1절부터 시작한다. 3절에서 잠시 멈춰 별도 터미널에 서버를 띄운 뒤 다음 셀로 간다. 측정값은 셀 출력에 표시되고 노트북을 저장하면 함께 보관된다.

## 1. 모델과 GPU 설정 읽기

모델이 글을 나눠 읽는 단위를 **토큰**, 글을 토큰 번호로 바꾸는 프로그램을 **토크나이저**라고 한다. **가중치**는 모델이 학습한 수다. 두 장비를 비교하려면 같은 모델 가중치와 토크나이저를 사용한다.

| 아래 코드의 이름 | 이 실습에서 뜻하는 것 |
| --- | --- |
| `MODEL` | 기본값은 `Qwen/Qwen3.8-27B`다. 양쪽 장비의 동일 모델 경로로 바꿀 수 있다. 기본 이름을 쓰면 필요한 파일을 Hugging Face에서 내려받는다. |
| `SINGLE_GPU`, `SERVER_GPUS` | 직접 실행과 서버가 사용할 GPU 번호다. 할당받은 GPU에 맞춘다. |
| `TENSOR_PARALLEL_SIZE` (TP) | 한 모델의 계산을 나눠 맡는 GPU 수다. 여기서 서버는 B300 한 장에서 실행한다(TP 1). |
| `NUM_PROMPTS`, `REPEATS` | 조건 하나당 질문 256개를 보내는 측정을 3회 한다. 도구가 보내는 준비 요청 8개는 별도다. |

BF16은 가중치 한 값을 2바이트로 저장한다. 270억 개라면 가중치만 약 54 GB다. 직접 실행에 쓸 GPU 한 장에는 가중치 외에도 중간 계산값과 캐시를 위한 여유가 필요하다. 서버와 부하를 보내는 클라이언트는 같은 머신에 둔다. [환경 준비 안내](README.md)

In [ ]:
import json
import os
import shlex
import subprocess
import sys
from urllib.request import Request, urlopen

MODEL = "Qwen/Qwen3.8-27B"
SINGLE_GPU = "0"
SERVER_GPUS = "0"
TENSOR_PARALLEL_SIZE = 1
BASE_URL = "http://127.0.0.1:8000"
NUM_PROMPTS = 256
REPEATS = 3

VLLM = [sys.executable, "-m", "vllm.entrypoints.cli.main"]

## 2. 같은 질문 묶음을 두 프로그램으로 실행하기

**프레임워크**인 PyTorch는 여러 수를 담은 배열(텐서)을 계산한다. **라이브러리**인 Transformers는 모델을 읽고 답변을 생성하는 코드를 제공한다. **추론 엔진**인 vLLM은 모델 실행에 더해 여러 요청을 언제 함께 계산할지도 정한다.

| 읽어 볼 파일 | 모델을 읽는 코드 | 답을 만드는 코드 |
| --- | --- | --- |
| [transformers_baseline.py](transformers_baseline.py) | `from_pretrained(...)` | `model.generate(inputs, ...)` |
| [vllm_baseline.py](vllm_baseline.py) | `LLM(...)` | `model.generate(prompts, ...)` |

두 셀 모두 **GPU 한 장**에 모델을 올린다. 길이 512토큰으로 맞춘 입력 8개를 한 배치로 주고 각 입력에서 128토큰을 생성한다. 출력 합계는 8 × 128 = **1,024토큰**이다. 반복 문장을 쓰는 이유는 입력 길이를 맞추기 위해서다.

**시간을 재는 구간**은 모델 로딩과 입력 준비가 끝난 뒤다. 같은 작업을 두 번 미리 실행하는 것을 워밍업이라 한다. 이후 세 번 시간을 잰다. PyTorch에서 `torch.cuda.synchronize()`는 GPU 계산이 끝날 때까지 기다리므로 CPU가 작업을 보낸 순간에 측정이 끝나는 일을 막는다.

**코드를 읽을 때** Transformers는 최소·최대 출력 길이를 모두 128로 맞춘다. vLLM은 종료 토큰을 만나도 계속 쓰게 하는 `ignore_eos=True`로 길이를 맞춘다. vLLM은 동시 처리 상한을 8로 두고, 접두사 재사용과 출력 번호를 글로 바꾸는 단계도 끈다.

Qwen 설정에 지정된 Transformers 클래스 이름은 `Qwen3_5ForConditionalGeneration`이다. Transformers는 이미지 처리 부분도 메모리에 올리고 vLLM은 텍스트 부분만 올린다. 여기서는 텍스트 생성 시간을 측정한다. 각 프로그램은 별도 프로세스에서 실행되고 종료할 때 GPU 메모리를 반환한다. [발표의 PyTorch 설명](chapter-04.html#pytorch)

In [ ]:
single_gpu_env = os.environ.copy()
single_gpu_env["CUDA_VISIBLE_DEVICES"] = SINGLE_GPU

result = subprocess.run(
    [sys.executable, "transformers_baseline.py", MODEL],
    env=single_gpu_env,
    stdout=subprocess.PIPE,
    text=True,
    check=True,
)
print("Transformers: elapsed_s, output_tokens, output_tokens_per_s")
print(result.stdout.split("elapsed_s, output_tokens, output_tokens_per_s\n", 1)[1])

In [ ]:
result = subprocess.run(
    [sys.executable, "vllm_baseline.py", MODEL],
    env=single_gpu_env,
    stdout=subprocess.PIPE,
    text=True,
    check=True,
)
print("vLLM: elapsed_s, output_tokens, output_tokens_per_s")
print(result.stdout.split("elapsed_s, output_tokens, output_tokens_per_s\n", 1)[1])

### 출력 한 줄 읽기

왼쪽부터 **경과 시간(초), 생성한 토큰 수, 초당 출력 토큰 수**다. 계산 예로 1,024토큰에 4초가 걸렸다면 1,024 ÷ 4 = 256토큰/초다. 이 숫자는 계산 방법을 보여 주는 예다. 실제 비교에는 바로 위 셀에 출력된 세 줄을 쓴다.

먼저 두 프로그램 모두 1,024토큰을 냈는지 보고, 그다음 세 번의 시간을 비교한다. 여기서 잰 것은 배치 전체가 끝나는 시간이다. 사용자가 첫 글을 보기까지의 시간은 뒤의 서버 실험에서 잰다.

두 프로그램은 사용하는 커널과 요청 처리 방식도 다르다. Qwen은 두 종류의 어텐션 계산을 함께 쓰므로, 이 측정만으로 특정 커널 하나가 시간 차이의 원인이라고 판정할 수 없다.

## 3. 요청을 받는 vLLM 서버 켜기

**서버**는 별도 프로그램에서 보낸 질문을 받아 답을 돌려준다. 여기서 질문을 보내는 **클라이언트**는 노트북이다. 둘은 같은 컴퓨터의 주소 `127.0.0.1`과 포트 `8000`을 통해 HTTP로 통신한다.

1. 아래 셀을 실행한다. 이 셀은 명령을 **출력**한다.
2. 출력된 한 줄을 같은 머신의 별도 터미널에 붙여 넣고 실행한다. 노트북과 같은 Python을 사용하도록 명령에 실행 파일 경로가 들어 있다.
3. 터미널에 `Application startup complete`가 뜨면 서버를 켜 둔 채 다음 셀을 실행한다.

TP 1에서는 B300 한 장이 한 모델을 처리한다. GPU 사이에 연산 결과를 전송할 필요가 없다. `--max-model-len 4096`은 요청 한 건의 입력과 출력 길이 합의 상한이다. `--max-num-seqs 32`는 한 단계에서 처리할 요청 수의 상한이고, `--max-num-batched-tokens 4096`은 그 단계에서 계산할 전체 토큰 수의 상한이다.

여러 질문의 앞부분이 같을 때 이전 계산을 다시 쓰는 기능을 **접두사 캐싱**이라고 한다. 이번 실험은 반복 입력도 다시 읽도록 이를 끈다. 이전 vLLM에서 옵션이 다르면 `vllm serve --help`에 표시된 이름을 쓴다(`--disable-prefix-caching` 또는 `--no-enable-prefix-caching`).

발표의 **NVIDIA Dynamo**는 워커가 여러 개일 때 어느 워커가 어느 요청을 받을지 정할 수 있다. 여기서는 vLLM 서버 하나의 처리량을 잰다. [Dynamo에서 라우터와 워커가 하는 일](chapter-04.html#dynamo)

In [ ]:
server_command = [
    "env", f"CUDA_VISIBLE_DEVICES={SERVER_GPUS}",
    *VLLM, "serve", MODEL,
    "--host", "127.0.0.1",
    "--port", "8000",
    "--tensor-parallel-size", str(TENSOR_PARALLEL_SIZE),
    "--dtype", "bfloat16",
    "--max-model-len", "4096",
    "--max-num-seqs", "32",
    "--max-num-batched-tokens", "4096",
    "--disable-prefix-caching",
    "--generation-config", "vllm",
    "--language-model-only",
]
print(shlex.join(server_command))

### 첫 질문 보내기

먼저 `/v1/models`로 서버가 제공하는 모델 이름을 읽는다. 다음으로 `/v1/chat/completions`에 한국어 질문을 보낸다. 슬래시로 시작하는 두 경로는 서버가 서로 다른 일을 받는 주소다. `messages`는 대화 내용이고 `enable_thinking=False`는 직접 답변을 요청하는 설정이다. 출력에서 모델 이름과 한국어 답변을 읽고 다음 셀로 넘어간다. 이 질문은 성능 측정에서 제외한다.

In [ ]:
with urlopen(f"{BASE_URL}/v1/models") as response:
    models = json.load(response)
print([item["id"] for item in models["data"]])

request = Request(
    f"{BASE_URL}/v1/chat/completions",
    data=json.dumps({
        "model": MODEL,
        "messages": [{
            "role": "user",
            "content": "추론 엔진과 HTTP 서버의 역할을 한국어로 두 문장으로 설명하세요.",
        }],
        "temperature": 0,
        "chat_template_kwargs": {"enable_thinking": False},
        "max_tokens": 128,
    }).encode("utf-8"),
    headers={"Content-Type": "application/json"},
)
with urlopen(request) as response:
    answer = json.load(response)
print(answer["choices"][0]["message"]["content"])

## 4. 부하 테스트 결과표 읽기

| 출력에서 찾을 이름 | 쉬운 뜻과 단위 |
| --- | --- |
| Successful / Failed requests | 끝까지 성공한 질문 수와 실패한 질문 수. 먼저 성공 256개, 실패 0개인지 본다. |
| Output token throughput | 서버 전체가 만든 출력 토큰 수 ÷ 측정 시간(초). 단위는 토큰/초다. |
| TTFT | 노트북 쪽에서 요청을 보낸 뒤 **첫 출력 토큰을 받을 때까지** 걸린 시간(ms). Gemma 실습의 서버 내부 TTFT보다 더 넓은 구간이다. |
| TPOT | 첫 출력 토큰 이후에 걸린 시간 ÷ 남은 출력 토큰 수(ms/토큰). 예를 들어 9개가 더 나오는 데 90 ms가 걸리면 10 ms/토큰이다. |
| E2EL | 요청 전송부터 **답변 전체를 받을 때까지** 걸린 시간(ms). |
| p50 / p95 | 빠른 순서로 놓았을 때 가운데 값 / 약 95%가 이 시간 안에 응답한 값. 느린 요청을 보려면 p95를 읽는다. |

아래 함수는 `vllm bench serve`로 토큰을 받는 동안 시간을 잰다. `random`은 길이를 맞춰 만든 입력이고, `--ignore-eos`는 답이 일찍 끝나도 정한 개수까지 생성하게 한다. 온도는 0이다. 채팅 양식을 적용하지 않는 `completions` 요청으로 속도를 측정하므로, 답변 내용이나 thinking 모드의 품질은 여기서 평가하지 않는다.

**동시 요청 상한 4**라면 최대 네 건까지 보낸 뒤, 하나가 끝나면 다음 요청을 보낸다. 초당 도착 건수를 따로 고정하지 않는다. 클라이언트가 자리가 나길 기다린 시간은 전송 전에 발생하므로 기본 TTFT·E2EL과 구분한다. 워밍업과 사전 확인 요청도 본 측정에서 빠진다.

예를 들어 출력이 요청당 128토큰이면 256건의 총 출력은 **32,768토큰**이다. 출력에서 성공 건수와 토큰 수를 먼저 확인한다. 입력 토큰은 도구가 번호를 문자열로 바꿨다가 다시 인코딩해 목표와 조금 다를 수 있다. 필요한 옵션은 `vllm bench serve --help`에서 확인할 수 있다. [발표의 지표 설명](chapter-04.html#metrics)

In [ ]:
def benchmark(input_tokens, output_tokens, concurrency):
    command = [
        *VLLM, "bench", "serve",
        "--backend", "openai",
        "--base-url", BASE_URL,
        "--model", MODEL,
        "--dataset-name", "random",
        "--random-input-len", str(input_tokens),
        "--random-output-len", str(output_tokens),
        "--random-range-ratio", "0",
        "--num-prompts", str(NUM_PROMPTS),
        "--num-warmups", "8",
        "--max-concurrency", str(concurrency),
        "--request-rate", "inf",
        "--temperature", "0",
        "--ignore-eos",
        "--percentile-metrics", "ttft,tpot,e2el",
        "--metric-percentiles", "50,95",
    ]
    for _ in range(REPEATS):
        result = subprocess.run(
            command, stdout=subprocess.PIPE, text=True, check=True,
        )
        print(
            f"input_tokens={input_tokens}, output_tokens={output_tokens}, "
            f"concurrency={concurrency}"
        )
        print(result.stdout.split("Serving Benchmark Result", 1)[1])

## 5. 동시에 보내는 질문 수 바꾸기

**바꾸는 값**은 동시 요청 상한 1, 4, 16, 32다. 입력 512토큰과 출력 128토큰은 같다. 각 조건을 세 번 측정한다.

먼저 `Output token throughput`을 읽고 같은 행의 p95 TTFT를 본다. 초당 생성량은 늘면서 첫 토큰을 기다리는 시간도 길어질 수 있다. **어디서 처리량 증가가 작아지고, 그때 느린 요청은 얼마나 기다렸는가?** 앞의 연속 배칭 예와 연결해 해석한다. 이 출력으로 GPU 사용률 자체를 읽을 수는 없다.

In [ ]:
for concurrency in [1, 4, 16, 32]:
    benchmark(input_tokens=512, output_tokens=128, concurrency=concurrency)

## 6. 질문 길이 바꾸기

**바꾸는 값**은 입력 128, 512, 2,048토큰이다. 동시 요청 상한은 4, 출력은 128토큰으로 고정하고 세 번씩 잰다.

모델이 질문을 읽고 첫 답변 토큰을 준비하는 계산을 **프리필(prefill)**이라고 한다. 질문이 길어지면 어떤 변화가 생길지 예상한 뒤 p50·p95 TTFT를 읽는다. 여기서 TTFT에는 네트워크와 서버 대기도 들어간다. 같은 길이에서도 p50과 p95가 얼마나 벌어지는지 확인한다. [질문 처리 순서](chapter-04.html#request)

In [ ]:
for input_tokens in [128, 512, 2048]:
    benchmark(input_tokens=input_tokens, output_tokens=128, concurrency=4)

## 7. 답변 길이 바꾸기

**바꾸는 값**은 출력 32, 128, 512토큰이다. 입력은 512토큰, 동시 요청 상한은 4로 고정하고 세 번씩 잰다.

첫 토큰 뒤에 다음 토큰을 이어서 만드는 단계를 **디코드(decode)**라고 한다. 먼저 답변 전체를 기다린 E2EL을 읽고, 다음으로 한 토큰을 더 만드는 평균 시간인 TPOT를 읽는다. **출력을 네 배로 늘렸을 때 전체 응답 시간과 토큰당 시간이 각각 몇 배가 됐는가?**

In [ ]:
for output_tokens in [32, 128, 512]:
    benchmark(input_tokens=512, output_tokens=output_tokens, concurrency=4)

## 8. H100과 B300 결과를 읽는 순서

동일한 모델·정밀도·입출력 길이·동시 요청 조건의 결과끼리 비교한다. 패키지 버전, 드라이버, CPU, GPU 연결 방식이 다르면 그 차이도 결과에 포함된다.

| 비교 | 해석 범위 |
| --- | --- |
| 직접 실행: H100 1개와 B300 1개 | 각 GPU 한 장으로 같은 입력 묶음을 끝내는 시간 |
| 서버: H100 4개(TP 4)와 B300 1개(TP 1) | GPU 수와 통신 비용까지 포함한 서버 구성 간 성능 |
| 같은 환경에서 동시 요청 수 변경 | 해당 요청 길이에서 처리량과 응답 지연의 변화 |
| 같은 환경에서 입력 또는 출력 길이 변경 | 해당 동시 요청 수에서 길이 변화의 영향 |

H100 4개 결과를 4로 나누어 H100 1개 성능으로 해석하지 않는다. TP 4는 GPU별 가중치 메모리와 연산을 나누지만 GPU 간 통신 비용도 발생한다. TP 4가 항상 빠르다고 가정하지 않는다. 이 설정으로 H100 또는 B300의 최대 성능을 측정했다고 주장하지 않는다.

먼저 같은 입력·출력 길이와 동시 요청 수의 두 행을 고른다. 각 행의 성공 건수부터 확인하고 다음 질문에 측정값과 단위를 넣어 답한다.

- 동일한 동시 요청 수에서 처리량과 p95 TTFT가 각각 어떻게 다른가?
- 동시 요청 수 증가에 따른 처리량 증가가 줄어드는 지점은 어디인가?
- 긴 입력과 긴 출력이 TTFT·TPOT·전체 응답시간에 미치는 영향은 어떻게 다른가?
- 세 번의 측정에서 차이가 일관되게 나타나는가?

반복 측정의 p95를 합쳐 전체 요청의 p95로 해석하지 않는다. 각 측정값과 그 변동 범위를 비교한다. 실제 성능 수치는 실행 후 판단한다.

## 9. 종료와 결과 읽기

서버를 실행한 터미널에서 `Ctrl+C`로 서버를 종료한다. 노트북을 닫는 것만으로 별도 터미널의 서버가 종료되지는 않는다.

셀 출력을 보관하려면 노트북을 저장한다. 이후 결과만 읽을 때는 실행 셀을 다시 실행하지 않는다. 새 측정은 회사 GPU를 다시 사용하며, 모델이 없으면 다운로드도 발생한다. 외부 유료 추론 API는 호출하지 않는다.

두 코드가 같은 모델로 답변을 생성하는 과정을 살펴봤다면, [Gemma 4와 DFlash 노트](vllm-gemma4-dflash.ipynb)에서 초안 모델을 추가했을 때의 차이를 확인할 수 있다. 이 후속 실습은 H100 네 장을 사용한다.

## 참고 자료

- [Inference Korea 5주차](https://songys.github.io/inference-korea/)
- [Inference Engineering 4장 한국어 참고 자료](https://corazzon.github.io/inference-engineering/ko/04-소프트웨어.html)
- [vLLM 서버 실행](https://docs.vllm.ai/en/stable/cli/serve/)
- [vLLM 서버 벤치마크](https://docs.vllm.ai/en/stable/cli/bench/serve/)
- [vLLM 병렬화](https://docs.vllm.ai/en/stable/serving/parallelism_scaling/)
- [Qwen3.8-27B 모델 카드](https://huggingface.co/Qwen/Qwen3.8-27B)
- [Qwen3.8-27B 모델 설정](https://huggingface.co/Qwen/Qwen3.8-27B/blob/main/config.json)
- [Qwen3.8-27B vLLM 실행 안내](https://recipes.vllm.ai/Qwen/Qwen3.8-27B)
- [PagedAttention 논문](https://arxiv.org/abs/2309.06180)

설명과 실습 코드는 별도로 작성했다. 모델 라이선스는 Apache-2.0이다. 이 노트북에는 교재 본문이나 그림을 복제하지 않았다.